# LifeMate — chọn model ASR phù hợp
**Đã điền ba input của bạn. Bật GPU + Internet (nếu luật cho phép), rồi Run All trong kernel mới.**
Mặc định: smoke 8 audio với 5 ứng viên chính, sau đó đổi `STAGE='benchmark'` để so cùng 100 audio. Chưa có kết quả chứng minh model nào thắng trên LifeMate.

| Bước | Chức năng |
|---|---|
| 1–3 | Cài thư viện, cấu hình, nạp mã dùng chung |
| 4–6 | Kiểm tra cache/audio, GPU, chuẩn bị lượt chạy |
| 7 | **PhoWhisper-medium**: ứng viên chuyên biệt tiếng Việt |
| 8 | **Whisper-large-v3-turbo**: chất lượng/tốc độ |
| 9–10 | **Qwen3-ASR-0.6B / 1.7B-hf**: so chất lượng và chi phí hai cỡ |
| 11 | **wav2vec2 tiếng Việt**: ứng viên CTC cỡ nhỏ |
| 12–14 | Whisper/PhoWhisper-small và PhoASR-small: tùy chọn, mặc định bỏ qua |
| 15–16 | Bảng chọn model và ZIP |

Giữ notebook/output **private** vì chứa transcript. Không đọc environment/tool labels, không gửi audio ra API. Reference chỉ dùng chấm điểm.


## 1. Cài thư viện
Chạy trước khi import model. Giữ nguyên torch/CUDA của Kaggle; offline thì đặt `INSTALL=False` và chuẩn bị packages trước.


In [ ]:
INSTALL = True
if INSTALL:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
        '--upgrade-strategy','only-if-needed',
        'transformers==5.14.1', 'huggingface-hub==1.24.0', 'tokenizers==0.22.2',
        'safetensors==0.8.0', 'soundfile==0.13.1', 'psutil==7.1.0','librosa==0.11.0'])
import subprocess, sys
DEPENDENCY_CHECK=subprocess.run([sys.executable,'-m','pip','check'],capture_output=True,text=True).stdout
print(DEPENDENCY_CHECK or 'pip check: không có nội dung báo cáo')
# Nếu đã import các thư viện này trước khi cài: restart kernel rồi Run All.


## 2. Cấu hình
Ba đường dẫn đã điền. Chỉ sửa nếu tên mount thay đổi. `benchmark` = 100 audio; `smoke` = 8; `validation` = 991 nếu cache hợp lệ.


In [ ]:
from pathlib import Path
DATASET_ROOT = Path('/kaggle/input/datasets/jamel27/zalo-aichallange-lifemate-dataset')
AUDIT_INPUT = Path('/kaggle/input/notebooks/jamel27/zalo-lifemate-audit')
FOLLOWUP_INPUT = Path('/kaggle/input/notebooks/jamel27/zalo-lifemate-audit-follow-up')
CACHE_DIR = None  # Tự lấy CSV trong FOLLOWUP_INPUT/lifemate_asr_followup hoặc ZIP follow-up.
DATASET_ZIP = None  # Chỉ đặt khi ZIP dataset có tên khác train+public_test.zip hoặc data.zip.
WORK_DIR = Path('/kaggle/working/lifemate_asr_inputs')  # Cache/audio chọn lọc; không đưa vào ZIP kết quả.
OUTPUT_DIR = Path('/kaggle/working/lifemate_asr_benchmark')
RESUME_DIR = None  # Output private cũ; lần đầu giữ None. Copy xong tự đặt lại None.
STAGE = 'smoke'  # Đạt smoke trước, rồi đổi benchmark (100) / validation (991).
DEVICE = 'cuda:0'
PRECISION = 'float16'  # CPU: DEVICE='cpu', PRECISION='float32'
OFFLINE = False
RUN_INFERENCE = True
RETRY_ERRORS = False
MAX_NEW_TOKENS = 400
MAX_LATENCY_P95_MS = None  # Điền SLA triển khai nếu đã biết; không tự đặt ngưỡng.
MAX_VRAM_GB = None  # Giới hạn PyTorch peak reserved trên máy đích, không phải tổng VRAM hệ thống.
CANDIDATES = [
    dict(name='phowhisper_medium',model_id='vinai/PhoWhisper-medium',family='seq2seq',revision='main',local_path=None),
    dict(name='whisper_turbo',model_id='openai/whisper-large-v3-turbo',family='seq2seq',revision='main',local_path=None),
    dict(name='qwen3_asr_06b',model_id='Qwen/Qwen3-ASR-0.6B-hf',family='qwen3_asr',revision='main',local_path=None,max_new_tokens=512),
    dict(name='qwen3_asr_17b',model_id='Qwen/Qwen3-ASR-1.7B-hf',family='qwen3_asr',revision='main',local_path=None,max_new_tokens=512),
    dict(name='whisper_small',model_id='openai/whisper-small',family='seq2seq',revision='main',local_path=None),
    dict(name='phowhisper_small',model_id='vinai/PhoWhisper-small',family='seq2seq',revision='main',local_path=None),
    dict(name='wav2vec2_vi',model_id='nguyenvulebinh/wav2vec2-base-vietnamese-250h',family='ctc',revision='main',local_path=None),
    dict(name='phoasr_small',model_id='Qualcomm-AI-Research/PhoASR-whisper-small',family='seq2seq',revision='main',local_path=None),
]
SELECTED_MODELS = ['phowhisper_medium','whisper_turbo','qwen3_asr_06b','qwen3_asr_17b','wav2vec2_vi']
# Bật model tùy chọn bằng cách thêm đúng name vào SELECTED_MODELS (đọc license trước khi chốt).
# Offline: mount đủ checkpoint/processor/tokenizer/config/weights, đặt local_path và OFFLINE=True.


## 3. Nạp mã dùng chung
Chỉ định nghĩa xử lý audio, scoring, resume và ZIP; **chưa tải/chạy model**. Cell dài có thể thu gọn.


In [ ]:
BENCHMARK_SOURCE = '"""LifeMate cached ASR benchmark. No dataset crawl, labels or external audio API."""\nfrom pathlib import Path\nimport json, hashlib, os, re, unicodedata, time, platform, threading, traceback, gc, shutil\nimport numpy as np\nimport pandas as pd\n\nNORMALIZATION = \'nfc_lower_punct_symbol_to_space_keep_accents_digits_v1\'\nCODE_VERSION = \'lifemate-model-selection-2.0\'\n\ndef normalize(text):\n    text = unicodedata.normalize(\'NFC\', str(text)).lower()\n    text = \'\'.join(\' \' if unicodedata.category(c)[0] in \'PS\' else c for c in text)\n    return \' \'.join(text.split())\n\ndef digest_file(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for block in iter(lambda: f.read(1024*1024), b\'\'): h.update(block)\n    return h.hexdigest()\n\ndef fingerprint(obj):\n    return hashlib.sha256(json.dumps(obj,sort_keys=True,ensure_ascii=False,default=str).encode()).hexdigest()\n\ndef safe_path(path):\n    p = Path(path)\n    if any(part.endswith(\'_environment\') for part in p.parts):\n        raise ValueError(\'Environment access forbidden: \'+str(p))\n    return p\n\ndef write_json(path, obj):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_suffix(path.suffix+\'.tmp\')\n    tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str));tmp.replace(path)\n\ndef append_json(path,obj):\n    with open(path,\'a\',encoding=\'utf-8\') as f:\n        f.write(json.dumps(obj,ensure_ascii=False,default=str,allow_nan=False)+\'\\n\');f.flush();os.fsync(f.fileno())\n\ndef read_journal(path):\n    """Recover only a torn final line; malformed interior records stop the run."""\n    path=Path(path)\n    if not path.exists(): return []\n    raw=path.read_bytes();lines=raw.splitlines(keepends=True);rows=[];offset=0\n    for i,line in enumerate(lines):\n        try: row=json.loads(line)\n        except (json.JSONDecodeError,UnicodeDecodeError):\n            if i!=len(lines)-1: raise RuntimeError(\'Corrupt journal interior: \'+str(path))\n            backup=path.with_name(path.name+\'.torn-\'+str(time.time_ns()))\n            backup.write_bytes(raw)\n            path.write_bytes(raw[:offset])\n            append_json(path.parent/\'recovery.jsonl\',{\'journal\':path.name,\'backup\':backup.name,\'discarded_bytes\':len(raw)-offset})\n            break\n        rows.append(row);offset+=len(line)\n    # A valid final JSON without newline must be terminated before appending.\n    if rows and path.read_bytes() and not path.read_bytes().endswith(b\'\\n\'):\n        with open(path,\'ab\') as f: f.write(b\'\\n\');f.flush();os.fsync(f.fileno())\n    return rows\n\ndef resolve_followup_cache(input_root,working_dir,override=None):\n    """Read the named follow-up mount only; never fall back to old audit metadata."""\n    import zipfile\n    root=safe_path(Path(input_root).resolve())\n    names=[\'references_enriched.csv\',\'split_manifest.csv\',\'benchmark_manifest.csv\']\n    candidates=[safe_path(Path(override).resolve())] if override is not None else [root/\'lifemate_asr_followup\',root]\n    for p in candidates:\n        safe_path(p)\n        if all((p/n).is_file() for n in names):\n            return p,dict(source=str(p),mode=\'csv_directory\')\n    if override is not None:\n        raise FileNotFoundError(\'CACHE_DIR must directly contain the three CSVs: \'+str(override))\n    archive=root/\'lifemate_asr_followup.zip\'\n    if not archive.is_file():\n        raise FileNotFoundError(\'Follow-up cache missing. Add the saved notebook output at \'+str(root)+\n            \'; expected lifemate_asr_followup/{\'+\',\'.join(names)+\'} or lifemate_asr_followup.zip.\')\n    with zipfile.ZipFile(archive) as z:\n        members={}\n        for info in z.infolist():\n            parts=info.filename.replace(\'\\\\\',\'/\').split(\'/\')\n            if info.is_dir() or \'__MACOSX\' in parts or any(x.endswith(\'_environment\') for x in parts): continue\n            if parts[-1] in names:\n                if any(x in [\'..\',\'\'] for x in parts) or info.filename.startswith(\'/\'):\n                    raise ValueError(\'Unsafe cache ZIP member\')\n                if parts[-1] in members: raise ValueError(\'Ambiguous cache CSV in ZIP: \'+parts[-1])\n                members[parts[-1]]=info\n        if set(members)!=set(names): raise FileNotFoundError(\'Follow-up ZIP must contain one of each required CSV\')\n        dest=safe_path(Path(working_dir).resolve()/\'followup_cache\');dest.mkdir(parents=True,exist_ok=True)\n        for name,info in members.items():\n            target=dest/name\n            if target.is_symlink(): raise ValueError(\'Symlink cache target forbidden\')\n            target.write_bytes(z.read(info))\n    return dest,dict(source=str(archive),mode=\'three_csvs_from_zip\')\n\n\ndef prepare_selected_audio(rows,dataset_root,working_dir,archive_path=None):\n    """Use selected files or extract only their exact entries from the dataset ZIP."""\n    import zipfile\n    root=safe_path(Path(dataset_root).resolve())\n    if all(resolve_audio(x,root).is_file() for x in rows.path):\n        return root,dict(source=str(root),mode=\'mounted_files\',n_selected=len(rows))\n    archives=[safe_path(Path(archive_path).resolve())] if archive_path is not None else [\n        p for p in [root/\'train+public_test.zip\',root/\'data.zip\'] if p.is_file()]\n    if len(archives)!=1:\n        raise FileNotFoundError(\'Selected audio missing under \'+str(root)+\n            \'. Mount extracted train+public_test/ or set DATASET_ZIP to the exact ZIP path.\')\n    archive=archives[0]\n    dest=safe_path(Path(working_dir).resolve()/\'selected_audio\');dest.mkdir(parents=True,exist_ok=True)\n    requested={resolve_audio(x,dest).relative_to(dest).as_posix() for x in rows.path}\n    with zipfile.ZipFile(archive) as z:\n        members={}\n        for info in z.infolist():\n            parts=info.filename.replace(\'\\\\\',\'/\').split(\'/\')\n            if info.is_dir() or \'__MACOSX\' in parts or any(x.endswith(\'_environment\') for x in parts): continue\n            if info.filename not in requested: continue\n            if info.filename in members: raise ValueError(\'Duplicate selected audio entry in ZIP\')\n            members[info.filename]=info\n        if set(members)!=requested:\n            raise FileNotFoundError(f\'ZIP missing {len(requested-set(members))} selected audio entries; expected train+public_test/... paths\')\n        required=sum(i.file_size for i in members.values())\n        if shutil.disk_usage(dest).free < required+16*1024*1024: raise OSError(\'Not enough space for selected audio\')\n        for name,info in members.items():\n            target=resolve_audio(name,dest)\n            target.parent.mkdir(parents=True,exist_ok=True)\n            tmp=target.with_suffix(target.suffix+\'.tmp\')\n            if tmp.is_symlink(): raise ValueError(\'Symlink extraction target forbidden\')\n            try:\n                with z.open(info) as src,open(tmp,\'wb\') as out: shutil.copyfileobj(src,out)\n                tmp.replace(target)\n            finally:\n                if tmp.exists(): tmp.unlink()\n    return dest,dict(source=str(archive),mode=\'selected_zip_entries\',n_selected=len(rows),bytes_selected=required)\n\n\ndef load_cache(cache_dir):\n    p=safe_path(cache_dir)\n    names=[\'references_enriched.csv\',\'split_manifest.csv\',\'benchmark_manifest.csv\']\n    for n in names:\n        if not (p/n).is_file(): raise FileNotFoundError(f\'Missing cache {p/n}; set CACHE_DIR to the directory containing the three CSV files.\')\n    r,s,b=[pd.read_csv(p/n,keep_default_na=False) for n in names]\n    for d in [r,s,b]:\n        if d.ref_uid.duplicated().any(): raise ValueError(\'Duplicate ref_uid\')\n    if not set(s.partition)<= {\'development\',\'validation\'}: raise ValueError(\'Unexpected partition\')\n    j=s[[\'ref_uid\',\'partition\',\'group_id\']].merge(r,on=\'ref_uid\',validate=\'one_to_one\',how=\'left\',indicator=True)\n    if not j._merge.eq(\'both\').all(): raise ValueError(\'Split IDs missing from reference cache\')\n    if not j.split.eq(\'train\').all() or not j.manifest.map(lambda x:Path(x).name==\'train.jsonl\').all():\n        raise ValueError(\'Public test or non-train manifest in labeled evaluation\')\n    # Compare by ID, never infer pairing by row order.\n    by=r.set_index(\'ref_uid\')\n    for d in [s,b]:\n        if not set(d.ref_uid)<=set(by.index): raise ValueError(\'Unmapped ID\')\n        for col in [\'path\',\'reference\',\'sample_uid\',\'kind\']:\n            if col in d and not (d[col].to_numpy()==by.loc[d.ref_uid,col].to_numpy()).all():\n                raise ValueError(\'Cache disagreement: \'+col)\n    dev=j[j.partition==\'development\'];val=j[j.partition==\'validation\']\n    if set(dev.sample_uid)&set(val.sample_uid) or set(dev.group_id)&set(val.group_id): raise ValueError(\'Group leakage\')\n    if not set(b.ref_uid)<=set(val.ref_uid): raise ValueError(\'Benchmark not validation-only\')\n    if len(r)!=6969 or len(s)!=5036 or len(val)!=991 or len(b)!=100: raise ValueError(\'Unexpected cache version; inspect explicitly before changing expected counts\')\n    if val.reference.map(lambda x:not isinstance(x,str) or not x.strip()).any(): raise ValueError(\'Missing gold text\')\n    for x in r.path: safe_path(x)\n    hashes={n:digest_file(p/n) for n in names}\n    return r,j.drop(columns=\'_merge\'),b,hashes\n\ndef select_rows(joined,benchmark,stage):\n    if stage==\'validation\': rows=joined[joined.partition==\'validation\'].copy().sort_values(\'ref_uid\')\n    elif stage in [\'smoke\',\'benchmark\']:\n        rows=joined.set_index(\'ref_uid\').loc[benchmark.ref_uid].reset_index()\n        if stage==\'smoke\':\n            # Deterministic endpoints per kind exercise short/long final/history; do not reshuffle benchmark.\n            picks=[]\n            for kind in [\'final\',\'history\']:\n                sub=rows[rows.kind==kind].sort_values([\'duration\',\'ref_uid\'])\n                picks+=sub.head(2).ref_uid.tolist()+sub.tail(2).ref_uid.tolist()\n            rows=rows[rows.ref_uid.isin(picks)].copy()\n    else: raise ValueError(\'stage must be smoke / benchmark / validation\')\n    return rows.reset_index(drop=True)\n\ndef resolve_audio(old_path,dataset_root):\n    old=safe_path(old_path)\n    parts=old.parts\n    if \'train+public_test\' not in parts: raise ValueError(\'No known root marker: \'+str(old))\n    tail=Path(*parts[parts.index(\'train+public_test\'):])\n    root=safe_path(dataset_root).resolve()\n    target=safe_path(root/tail).resolve()\n    if not target.is_relative_to(root): raise ValueError(\'Audio escaped configured root\')\n    safe_path(target)\n    return target\n\ndef runtime_info():\n    import torch,transformers,soundfile,psutil,importlib.metadata as md\n    versions={n:md.version(n) for n in [\'torch\',\'transformers\',\'numpy\',\'pandas\',\'soundfile\',\'psutil\',\'huggingface-hub\',\'tokenizers\',\'safetensors\']}\n    return dict(versions=versions,python=platform.python_version(),platform=platform.platform(),cpu=platform.processor(),\n                cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,\n                gpu_total_bytes=torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None,\n                cpu_threads=torch.get_num_threads())\n\ndef resolve_checkpoint(spec,out,offline=False):\n    """Pin remote branch to SHA once; local mounted checkpoint gets content fingerprint."""\n    from huggingface_hub import HfApi,snapshot_download\n    lock=out/(spec[\'name\']+\'_checkpoint.json\')\n    old=json.loads(lock.read_text()) if lock.exists() else None\n    if spec.get(\'local_path\'):\n        source=safe_path(spec[\'local_path\']).resolve()\n        if not source.is_dir(): raise FileNotFoundError(\'Checkpoint mount missing: \'+str(source))\n        if not (source/\'config.json\').is_file(): raise ValueError(\'Local checkpoint must contain config.json; no scan performed\')\n        entries=[]\n        # Only checkpoint files, never dataset audio or environment directories.\n        for root,dirs,files in os.walk(source):\n            dirs[:]=sorted(d for d in dirs if not d.endswith(\'_environment\') and d not in [\'.git\',\'.cache\'])\n            for name in sorted(files):\n                f=safe_path((Path(root)/name).resolve())\n                if f.suffix in [\'.json\',\'.bin\',\'.safetensors\',\'.txt\',\'.model\',\'.jinja\']:\n                    entries.append((str(f.relative_to(source)),digest_file(f)))\n        if not entries or not (source/\'config.json\').exists(): raise ValueError(\'Incomplete local checkpoint\')\n        identity=\'local-sha256:\'+fingerprint(entries)\n        obj=dict(model_id=spec[\'model_id\'],revision=identity,requested_revision=spec.get(\'revision\',\'main\'),local_files=entries)\n    else:\n        if offline and not old: raise ValueError(\'OFFLINE requires local_path or a previously saved checkpoint lock and HF cache\')\n        sha=old[\'revision\'] if old and old.get(\'model_id\')==spec[\'model_id\'] and old.get(\'requested_revision\')==spec.get(\'revision\',\'main\') else None\n        if offline and sha is None: raise ValueError(\'Checkpoint lock does not match offline request\')\n        if sha is None: sha=HfApi().model_info(spec[\'model_id\'],revision=spec.get(\'revision\',\'main\')).sha\n        source=Path(snapshot_download(spec[\'model_id\'],revision=sha,local_files_only=offline,\n                allow_patterns=[\'*.json\',\'*.txt\',\'*.model\',\'*.jinja\',\'*.safetensors\',\'pytorch_model*.bin\']))\n        identity=sha;obj=dict(model_id=spec[\'model_id\'],revision=sha,requested_revision=spec.get(\'revision\',\'main\'))\n    write_json(lock,obj)\n    return source,identity\n\nclass RSSMonitor:\n    """Sample process RSS every 10 ms; brief peaks may be missed; not system RAM."""\n    def __enter__(self):\n        import psutil\n        self.process=psutil.Process();self.peak=self.process.memory_info().rss;self.stop=threading.Event()\n        def sample():\n            while not self.stop.wait(.01): self.peak=max(self.peak,self.process.memory_info().rss)\n        self.thread=threading.Thread(target=sample,daemon=True);self.thread.start();return self\n    def __exit__(self,*args):\n        self.peak=max(self.peak,self.process.memory_info().rss);self.stop.set();self.thread.join()\n\ndef whisper_sequence_diagnostics(sequence,eos_token_id,pad_token_id=None):\n    """Check raw generate sequences, not Whisper\'s EOS-stripped segment tokens."""\n    tokens=sequence.tolist() if hasattr(sequence,\'tolist\') else list(sequence)\n    eos=set(eos_token_id if isinstance(eos_token_id,(list,tuple,set)) else [eos_token_id])\n    if None in eos or not eos: raise ValueError(\'Whisper EOS token is not configured\')\n    # Some generation configurations use a separate pad token after EOS.\n    while tokens and tokens[-1]==pad_token_id and pad_token_id not in eos: tokens.pop()\n    return dict(output_token_count=len(tokens),ended_with_eos=bool(tokens and tokens[-1] in eos))\n\n\nclass ASR:\n    def __init__(self,spec,source,revision,device=\'cuda:0\',precision=\'float16\',max_new_tokens=400):\n        import torch\n        self.spec=spec;self.revision=revision;self.device=device;self.family=spec[\'family\'];self.max_new_tokens=max_new_tokens\n        self.dtype=torch.float16 if precision==\'float16\' else torch.float32\n        if device.startswith(\'cuda\') and not torch.cuda.is_available(): raise RuntimeError(\'Requested CUDA unavailable; select Kaggle GPU or explicitly set DEVICE=cpu and PRECISION=float32\')\n        if device==\'cpu\' and precision!=\'float32\': raise ValueError(\'Use float32 on CPU\')\n        if self.family==\'seq2seq\':\n            from transformers import WhisperProcessor as proc,WhisperForConditionalGeneration as model\n        elif self.family==\'ctc\':\n            from transformers import Wav2Vec2Processor as proc,Wav2Vec2ForCTC as model\n        elif self.family==\'qwen3_asr\':\n            from transformers import AutoProcessor as proc,Qwen3ASRForConditionalGeneration as model\n        else: raise ValueError(\'Unsupported model family: \'+str(self.family))\n        self.processor=proc.from_pretrained(str(source),local_files_only=True)\n        self.model=model.from_pretrained(str(source),local_files_only=True,dtype=self.dtype,attn_implementation=\'eager\').to(device).eval()\n        if self.family==\'seq2seq\': self.model.generation_config.forced_decoder_ids=None\n    def sync(self):\n        import torch\n        if self.device.startswith(\'cuda\'): torch.cuda.synchronize(self.device)\n    def transcribe(self,audio_path):\n        """Only audio enters inference. No reference, context, tool schema or confidence."""\n        import torch,soundfile as sf\n        result=dict(status=\'error\',text_model=\'\',text_normalized=\'\',model_id=self.spec[\'model_id\'],revision=self.revision,\n                    latency_ms=None,model_ms=None,duration_s=None,rtf=None,warnings=[],error_code=None)\n        self.sync();start=time.perf_counter();mon=None\n        try:\n            if self.device.startswith(\'cuda\'): torch.cuda.reset_peak_memory_stats(self.device)\n            with RSSMonitor() as mon:\n                wav,sr=sf.read(safe_path(audio_path),dtype=\'float32\',always_2d=True)\n                if sr!=16000 or wav.shape[1]!=1: raise ValueError(\'UNEXPECTED_AUDIO_FORMAT: expected mono 16000Hz\')\n                wav=wav[:,0];duration=len(wav)/sr;result[\'duration_s\']=duration\n                if not len(wav) or not np.isfinite(wav).all(): raise ValueError(\'INVALID_AUDIO\')\n                if self.family==\'seq2seq\' and duration>30: raise ValueError(\'LONG_AUDIO_UNSUPPORTED: no truncation permitted\')\n                if self.family==\'ctc\' and duration>10: result[\'warnings\'].append(\'above_model_card_10s_recommendation_full_audio_used\')\n                if self.family==\'seq2seq\':\n                    batch=self.processor(wav,sampling_rate=sr,return_tensors=\'pt\',return_attention_mask=True,truncation=False)\n                    inputs={k:v.to(self.device,dtype=self.dtype) if k==\'input_features\' else v.to(self.device) for k,v in batch.items()}\n                elif self.family==\'ctc\':\n                    batch=self.processor(wav,sampling_rate=sr,return_tensors=\'pt\',padding=False)\n                    inputs={k:v.to(self.device,dtype=self.dtype) if k==\'input_values\' else v.to(self.device) for k,v in batch.items()}\n                elif self.family==\'qwen3_asr\':\n                    # Native HF processor accepts the waveform already validated above.\n                    # Language is fixed globally; no transcript/context/hotwords are passed.\n                    # Pinned native processor defaults to 16 kHz and PyTorch tensors.\n                    batch=self.processor.apply_transcription_request(audio=wav,language=\'vi\')\n                    inputs={k:v.to(device=self.device,dtype=self.dtype) if v.is_floating_point() else v.to(self.device) for k,v in batch.items()}\n                self.sync();model_start=time.perf_counter()\n                with torch.inference_mode():\n                    if self.family==\'seq2seq\':\n                        # HF Whisper\'s default tensor output strips EOS during segment processing.\n                        # The structured short-form output retains raw sequences including EOS.\n                        generated=self.model.generate(**inputs,language=\'vi\',task=\'transcribe\',num_beams=1,do_sample=False,\n                                                max_new_tokens=self.max_new_tokens,return_timestamps=False,\n                                                return_dict_in_generate=True,output_scores=False,output_logits=False)\n                        ids=generated.sequences\n                    elif self.family==\'ctc\': ids=self.model(**inputs).logits.argmax(dim=-1)\n                    else:\n                        generated=self.model.generate(**inputs,max_new_tokens=self.max_new_tokens,num_beams=1,do_sample=False,\n                            return_dict_in_generate=True,output_scores=False,output_logits=False)\n                        ids=generated.sequences[:,inputs[\'input_ids\'].shape[1]:]\n                self.sync();result[\'model_ms\']=(time.perf_counter()-model_start)*1000\n                if self.family==\'qwen3_asr\':\n                    raw=self.processor.decode(ids,return_format=\'raw\')\n                    parsed=self.processor.decode(ids,return_format=\'transcription_only\')\n                    result[\'text_output_raw\']=raw[0] if isinstance(raw,list) else raw\n                    text=parsed[0] if isinstance(parsed,list) else parsed\n                    if \'<asr_text>\' not in result[\'text_output_raw\']: raise RuntimeError(\'QWEN_OUTPUT_SCHEMA_MISSING_ASR_TEXT\')\n                else: text=self.processor.batch_decode(ids,skip_special_tokens=True)[0]\n                result[\'text_model\']=text;result[\'text_normalized\']=normalize(text)\n                if self.family in [\'seq2seq\',\'qwen3_asr\']:\n                    diagnostics=whisper_sequence_diagnostics(ids[0],self.model.generation_config.eos_token_id,\n                        self.model.generation_config.pad_token_id)\n                    result.update(diagnostics)\n                    # Keep partial text for diagnosis, but never score a genuinely unfinished sequence as success.\n                    if not diagnostics[\'ended_with_eos\']: raise RuntimeError(\'TOKEN_LIMIT_OR_MISSING_EOS\')\n                if not text.strip(): raise RuntimeError(\'EMPTY_TRANSCRIPT\')\n                result[\'status\']=\'ok\'\n        except Exception as exc:\n            result[\'error_code\']=type(exc).__name__+\': \'+str(exc)\n        finally:\n            self.sync();result[\'latency_ms\']=(time.perf_counter()-start)*1000\n            if result[\'duration_s\']: result[\'rtf\']=result[\'latency_ms\']/1000/result[\'duration_s\']\n            result[\'rss_peak_bytes\']=mon.peak if mon else None\n            result[\'vram_peak_allocated_bytes\']=torch.cuda.max_memory_allocated(self.device) if self.device.startswith(\'cuda\') else None\n            result[\'vram_peak_reserved_bytes\']=torch.cuda.max_memory_reserved(self.device) if self.device.startswith(\'cuda\') else None\n        return result\n\ndef run_benchmark(rows,specs,cfg,cache_hashes):\n    import torch\n    out=safe_path(cfg[\'output_dir\']);out.mkdir(parents=True,exist_ok=True)\n    env=runtime_info();write_json(out/\'runtime.json\',env)\n    torch.manual_seed(2026);np.random.seed(2026)\n    # preflight only selected audio paths, no decoding/hash/scan; fail clearly on missing mounts.\n    if not safe_path(cfg[\'dataset_root\']).is_dir(): raise FileNotFoundError(\'DATASET_ROOT missing: \'+str(cfg[\'dataset_root\']))\n    paths={x.ref_uid:str(resolve_audio(x.path,cfg[\'dataset_root\'])) for x in rows.itertuples()}\n    print(\'Selected audio paths missing:\',sum(not Path(p).is_file() for p in paths.values()),\'(each is recorded as a sample error)\')\n    write_json(out/\'selection.json\',dict(stage=cfg[\'stage\'],ref_uids=rows.ref_uid.tolist(),cache_hashes=cache_hashes))\n    run_dirs=[]\n    for spec in specs:\n        engine=None\n        try:\n            source,revision=resolve_checkpoint(spec,out,cfg.get(\'offline\',False))\n            token_limit=spec.get(\'max_new_tokens\',cfg[\'max_new_tokens\'])\n            identity=dict(code=CODE_VERSION,source_sha256=cfg[\'source_sha256\'],model_id=spec[\'model_id\'],revision=revision,\n                          family=spec[\'family\'],device=cfg[\'device\'],precision=cfg[\'precision\'],max_new_tokens=token_limit,\n                          decoding=\'greedy_vi_no_reference_no_context_no_external_LM\',normalization=NORMALIZATION,cache_hashes=cache_hashes,\n                          runtime=env)\n            run_id=fingerprint(identity)[:20];rd=out/(spec[\'name\']+\'_\'+run_id);rd.mkdir(exist_ok=True);run_dirs.append(rd)\n            write_json(rd/\'config.json\',identity);journal=rd/\'predictions.jsonl\'\n            prev=read_journal(journal)\n            latest={x[\'ref_uid\']:x for x in prev}\n            completed={uid for uid,x in latest.items() if x[\'status\']==\'ok\' or not cfg.get(\'retry_errors\',False)}\n            pending=rows[~rows.ref_uid.isin(completed)]\n            if pending.empty: print(spec[\'name\'],\'already completed\');continue\n            with RSSMonitor() as load_mon:\n                start=time.perf_counter();engine=ASR(spec,source,revision,cfg[\'device\'],cfg[\'precision\'],token_limit);engine.sync()\n                load_ms=(time.perf_counter()-start)*1000\n            write_json(rd/\'model_config.json\',engine.model.config.to_dict())\n            if getattr(engine.model,\'generation_config\',None) is not None:\n                write_json(rd/\'generation_config.json\',engine.model.generation_config.to_dict())\n            session_id=str(time.time_ns())\n            write_json(rd/(\'session_\'+session_id+\'.json\'),dict(session_id=session_id,load_ms=load_ms,load_rss_peak_bytes=load_mon.peak,hardware=env))\n            warm=engine.transcribe(paths[pending.iloc[0].ref_uid]);write_json(rd/(\'warmup_\'+session_id+\'.json\'),warm)\n            if warm[\'status\']!=\'ok\': print(\'Warmup failed (retained):\',warm[\'error_code\'])\n            for i,row in enumerate(pending.itertuples()):\n                record=engine.transcribe(paths[row.ref_uid])\n                if record.get(\'duration_s\') is not None and abs(record[\'duration_s\']-float(row.duration)) > 1/16000:\n                    record[\'status\']=\'error\';record[\'error_code\']=\'CACHE_AUDIO_DURATION_MISMATCH\'\n                record.update(ref_uid=row.ref_uid,session_id=session_id,run_id=run_id)\n                append_json(journal,record)\n                if i%10==0: print(spec[\'name\'],i+1,\'/\',len(pending),record[\'status\'])\n                if record[\'status\']!=\'ok\' and \'out of memory\' in str(record[\'error_code\']).lower(): torch.cuda.empty_cache()\n        except Exception:\n            err=dict(model=spec[\'name\'],stage=cfg[\'stage\'],traceback=traceback.format_exc(),time_ns=time.time_ns())\n            append_json(out/\'model_errors.jsonl\',err);print(\'MODEL ERROR:\',spec[\'name\'],err[\'traceback\'])\n        finally:\n            del engine;gc.collect()\n            if torch.cuda.is_available(): torch.cuda.empty_cache()\n    return run_dirs\n\ndef edit_distance(a,b):\n    prev=list(range(len(b)+1))\n    for i,x in enumerate(a,1):\n        cur=[i]\n        for j,y in enumerate(b,1): cur.append(min(cur[-1]+1,prev[j]+1,prev[j-1]+(x!=y)))\n        prev=cur\n    return prev[-1]\n\ndef score_rows(rows,records):\n    latest={x[\'ref_uid\']:x for x in records};out=[]\n    for row in rows.itertuples():\n        p=latest.get(row.ref_uid);status=p[\'status\'] if p else \'missing\'\n        ref=normalize(row.reference);hyp=normalize(p[\'text_model\']) if p and status==\'ok\' else \'\'\n        # Failure-as-empty keeps every selected reference in the denominator; also report success-only.\n        rc=ref.replace(\' \',\'\');hc=hyp.replace(\' \',\'\')\n        out.append(dict(ref_uid=row.ref_uid,sample_uid=row.sample_uid,kind=row.kind,duration=float(row.duration),status=status,\n                        reference=row.reference,text_model=p.get(\'text_model\',\'\') if p else \'\',word_edits=edit_distance(ref.split(),hyp.split()),\n                        n_words=len(ref.split()),char_edits=edit_distance(rc,hc),n_chars=len(rc),\n                        latency_ms=p.get(\'latency_ms\') if p else None,rtf=p.get(\'rtf\') if p else None,\n                        rss_peak_bytes=p.get(\'rss_peak_bytes\') if p else None,\n                        vram_peak_allocated_bytes=p.get(\'vram_peak_allocated_bytes\') if p else None,\n                        vram_peak_reserved_bytes=p.get(\'vram_peak_reserved_bytes\') if p else None,\n                        warnings=json.dumps(p.get(\'warnings\',[])) if p else \'[]\'))\n    return pd.DataFrame(out)\n\ndef aggregate(d):\n    ok=d[d.status==\'ok\'];timed=ok.dropna(subset=[\'latency_ms\'])\n    return dict(n=len(d),n_ok=len(ok),n_error=int(d.status.eq(\'error\').sum()),n_missing=int(d.status.eq(\'missing\').sum()),\n        complete=bool(not d.status.eq(\'missing\').any()),failure_rate=float(d.status.ne(\'ok\').mean()),\n        wer=float(d.word_edits.sum()/d.n_words.sum()) if len(ok) and d.n_words.sum() else None,\n        cer_no_spaces=float(d.char_edits.sum()/d.n_chars.sum()) if len(ok) and d.n_chars.sum() else None,\n        wer_success_only=float(ok.word_edits.sum()/ok.n_words.sum()) if ok.n_words.sum() else None,\n        cer_success_only=float(ok.char_edits.sum()/ok.n_chars.sum()) if ok.n_chars.sum() else None,\n        latency_p50_ms=float(timed.latency_ms.median()) if len(timed) else None,\n        latency_p95_ms=float(timed.latency_ms.quantile(.95)) if len(timed) else None,\n        rtf_weighted=float((timed.latency_ms.sum()/1000)/timed.duration.sum()) if len(timed) else None,\n        rss_peak_bytes=int(ok.rss_peak_bytes.max()) if ok.rss_peak_bytes.notna().any() else None,\n        vram_peak_allocated_bytes=int(ok.vram_peak_allocated_bytes.max()) if ok.vram_peak_allocated_bytes.notna().any() else None,\n        vram_peak_reserved_bytes=int(ok.vram_peak_reserved_bytes.max()) if ok.vram_peak_reserved_bytes.notna().any() else None)\n\ndef evaluate(rows,out_dir,run_dirs=None):\n    out=Path(out_dir);summaries=[];scored={}\n    # Enumerate only immediate benchmark output children, never dataset.\n    for rd in sorted(out.iterdir() if run_dirs is None else set(map(Path,run_dirs))):\n        if not rd.is_dir() or not (rd/\'config.json\').is_file(): continue\n        d=score_rows(rows,read_journal(rd/\'predictions.jsonl\'));d.to_csv(rd/\'per_audio.csv\',index=False);scored[rd.name]=d\n        groups={\'all\':d,\'final\':d[d.kind==\'final\'],\'history\':d[d.kind==\'history\'],\n                \'duration_le10s\':d[d.duration<=10],\'duration_gt10s\':d[d.duration>10]}\n        # Human annotations are read only AFTER inference; hints are not gold categories.\n        ann=out/\'dialect_annotations.csv\'\n        if ann.exists():\n            labels=pd.read_csv(ann,keep_default_na=False)\n            if labels.ref_uid.duplicated().any(): raise ValueError(\'Duplicate annotation IDs; adjudicate first\')\n            # Annotation templates also carry duration/kind/sample_uid for human review.\n            # Preserve scoring metadata from the validated rows when joining labels.\n            merged=d.merge(labels,on=\'ref_uid\',how=\'left\',validate=\'one_to_one\',suffixes=(\'\', \'_annotation\'))\n            for col in [\'accent_region\',\'accent_variation\',\'dialect_vocabulary\',\'vi_accented_english\',\'code_switching\',\'noise\']:\n                if col in merged:\n                    merged[col]=merged[col].fillna(\'unannotated\').replace(\'\', \'unannotated\')\n                    for label,sub in merged.groupby(col,dropna=False): groups[f\'{col}={label}\']=sub\n        for name,sub in groups.items():\n            if len(sub): summaries.append(dict(run=rd.name,subset=name,**aggregate(sub)))\n    summary=pd.DataFrame(summaries);summary.to_csv(out/\'metrics.csv\',index=False)\n    return summary,scored\n\n\ndef selection_table(summary,scored,out_path,max_latency_p95_ms=None,max_vram_gb=None):\n    """Describe eligible measured tradeoffs; never declare a winner without human review."""\n    records=[]\n    if not summary.empty:\n        for row in summary[summary.subset==\'all\'].to_dict(\'records\'):\n            eligible=row[\'complete\'] and row[\'n_ok\']==row[\'n\'] and row[\'n\']>0 and pd.notna(row[\'wer\'])\n            reasons=[]\n            if not eligible: reasons.append(\'incomplete_or_errors\')\n            if max_latency_p95_ms is not None:\n                if pd.isna(row[\'latency_p95_ms\']) or row[\'latency_p95_ms\']>max_latency_p95_ms:\n                    eligible=False;reasons.append(\'latency_limit_or_missing_measurement\')\n            if max_vram_gb is not None:\n                peak=row[\'vram_peak_reserved_bytes\']\n                if pd.isna(peak) or peak>max_vram_gb*1024**3:\n                    eligible=False;reasons.append(\'vram_limit_or_missing_measurement\')\n            records.append(dict(**row,eligible=bool(eligible),excluded_reason=\';\'.join(reasons),human_review_required=True))\n    result=pd.DataFrame(records)\n    if not result.empty:\n        result[\'pareto_wer_latency\']=False\n        eligible=result[result.eligible & result.latency_p95_ms.notna()]\n        for i,a in eligible.iterrows():\n            dominated=any((other.wer<=a.wer and other.latency_p95_ms<=a.latency_p95_ms and\n                (other.wer<a.wer or other.latency_p95_ms<a.latency_p95_ms)) for _,other in eligible.iterrows())\n            result.loc[i,\'pareto_wer_latency\']=not dominated\n        result=result.sort_values([\'eligible\',\'wer\',\'latency_p95_ms\'],ascending=[False,True,True])\n    result.to_csv(out_path,index=False)\n    return result\n\ndef paired_bootstrap(scored,out_path,n_boot=2000,seed=2026):\n    """Cluster paired bootstrap by conversation, conditional on fixed evaluation subset."""\n    rng=np.random.default_rng(seed);names=sorted(scored);results=[]\n    for i,a in enumerate(names):\n        for b in names[i+1:]:\n            x,y=scored[a],scored[b]\n            if not set(x.ref_uid)==set(y.ref_uid) or x.status.eq(\'missing\').any() or y.status.eq(\'missing\').any(): continue\n            if not x.status.eq(\'ok\').any() or not y.status.eq(\'ok\').any(): continue\n            merged=x[[\'ref_uid\',\'sample_uid\',\'word_edits\',\'n_words\']].merge(y[[\'ref_uid\',\'word_edits\']],on=\'ref_uid\',suffixes=(\'_a\',\'_b\'),validate=\'one_to_one\')\n            groups=merged.groupby(\'sample_uid\')[[\'word_edits_a\',\'word_edits_b\',\'n_words\']].sum().to_numpy()\n            if len(groups)<2: continue\n            deltas=[]\n            for _ in range(n_boot):\n                sampled=groups[rng.integers(0,len(groups),len(groups))].sum(axis=0)\n                deltas.append((sampled[0]-sampled[1])/sampled[2])\n            results.append(dict(a=a,b=b,n_conversations=len(groups),delta_wer_a_minus_b=float((groups[:,0].sum()-groups[:,1].sum())/groups[:,2].sum()),\n                                ci95_low=float(np.quantile(deltas,.025)),ci95_high=float(np.quantile(deltas,.975)),n_boot=n_boot))\n    pd.DataFrame(results,columns=[\'a\',\'b\',\'n_conversations\',\'delta_wer_a_minus_b\',\'ci95_low\',\'ci95_high\',\'n_boot\']).to_csv(out_path,index=False)\n    return results\n\ndef save_template_preserving_rows(new,path):\n    if path.exists():\n        previous=pd.read_csv(path,keep_default_na=False)\n        if previous.ref_uid.duplicated().any(): raise ValueError(\'Duplicate annotation IDs\')\n        new=pd.concat([previous,new[~new.ref_uid.isin(previous.ref_uid)]],ignore_index=True)\n    new.to_csv(path,index=False)\n\ndef annotation_templates(rows,out_dir,seed=2026):\n    out=Path(out_dir)\n    ann=rows[[\'ref_uid\',\'sample_uid\',\'kind\',\'path\',\'duration\']].copy()\n    ann[\'selection_reason\']=\'fixed_benchmark\' if len(rows)<=100 else \'fixed_validation\'\n    for col in [\'annotator_id\',\'accent_region\',\'accent_variation\',\'dialect_vocabulary\',\'vi_accented_english\',\'code_switching\',\'noise\',\'reference_problem\',\'notes\']:\n        ann[col]=\'\'\n    save_template_preserving_rows(ann, out/\'dialect_annotations.csv\')\n    gold=rows[[\'ref_uid\',\'reference\']].copy()\n    for cat in [\'person\',\'money\',\'time\',\'location\',\'device\',\'negation\']: gold[cat+\'_spans_json\']=\'\'\n    save_template_preserving_rows(gold, out/\'critical_gold_annotation.csv\')\n\ndef make_review(scored,out_dir):\n    records=[]\n    for name,d in scored.items():\n        for row in d.itertuples():\n            r=dict(model=name,ref_uid=row.ref_uid,status=row.status,reference=row.reference,hypothesis=row.text_model)\n            for cat in [\'person\',\'money\',\'time\',\'location\',\'device\',\'negation\']:r[cat+\'_judgment\']=\'\'\n            r[\'notes\']=\'\';records.append(r)\n    # Never overwrite filled human review.\n    path=Path(out_dir)/\'critical_model_review_template.csv\'\n    new=pd.DataFrame(records,columns=[\'model\',\'ref_uid\',\'status\',\'reference\',\'hypothesis\',\n        \'person_judgment\',\'money_judgment\',\'time_judgment\',\'location_judgment\',\'device_judgment\',\'negation_judgment\',\'notes\'])\n    if path.exists():\n        old=pd.read_csv(path,keep_default_na=False)\n        keys=[\'model\',\'ref_uid\']\n        if old.duplicated(keys).any(): raise ValueError(\'Duplicate model review IDs\')\n        # Preserve the reviewed hypothesis and judgments, including after retry.\n        # A changed hypothesis needs explicit human review rather than silent replacement.\n        seen=set(map(tuple,old[keys].to_numpy()))\n        new=pd.concat([old,new[[tuple(k) not in seen for k in new[keys].to_numpy()]]],ignore_index=True)\n    new.to_csv(path,index=False)\n\ndef package_outputs(out_dir,zip_base):\n    """Export comparison artifacts only; standalone and usable after interrupted runs."""\n    from pathlib import Path\n    import hashlib, json, zipfile\n    root=Path(out_dir).resolve()\n    if not root.is_dir(): raise FileNotFoundError(root)\n    target=Path(str(zip_base)+\'.zip\').resolve()\n    target.parent.mkdir(parents=True,exist_ok=True)\n    shared={\'runtime.json\',\'selection.json\',\'benchmark_request.json\',\'benchmark_source.py\',\'input_provenance.json\',\n            \'scoring_patch.json\',\'scoring_source.py\',\n            \'metrics.csv\',\'paired_bootstrap.csv\',\'comparison_status.csv\',\'model_selection.csv\',\'dependency_check.txt\',\n            \'dialect_annotations.csv\',\'critical_gold_annotation.csv\',\n            \'critical_model_review_template.csv\',\'model_errors.jsonl\'}\n    run_files={\'config.json\',\'model_config.json\',\'generation_config.json\',\n               \'predictions.jsonl\',\'per_audio.csv\',\'recovery.jsonl\'}\n    selected=[]\n    for p in sorted(root.iterdir()):\n        if p.is_symlink(): continue\n        if p.is_file() and (p.name in shared or\n                (p.name.startswith(\'selection_\') and p.suffix==\'.csv\') or\n                p.name.endswith(\'_checkpoint.json\')):\n            selected.append(p)\n        elif p.is_dir() and not p.name.endswith(\'_environment\') and (p/\'config.json\').is_file():\n            for f in sorted(p.iterdir()):\n                if f.is_symlink() or not f.is_file(): continue\n                if f.name in run_files or (f.name.startswith((\'session_\',\'warmup_\')) and f.suffix==\'.json\') or f.name.startswith(\'predictions.jsonl.torn-\'):\n                    selected.append(f)\n    entries=[]\n    tmp=target.with_suffix(\'.zip.tmp\')\n    try:\n        with zipfile.ZipFile(tmp,\'w\',compression=zipfile.ZIP_DEFLATED) as archive:\n            for f in selected:\n                relative=f.relative_to(root).as_posix()\n                data=f.read_bytes()\n                entries.append(dict(path=relative,bytes=len(data),sha256=hashlib.sha256(data).hexdigest()))\n                archive.writestr(relative,data)\n            manifest=dict(schema_version=1,private=True,files=entries,\n                missing_summary_files=[n for n in [\'benchmark_request.json\',\'metrics.csv\',\'comparison_status.csv\'] if not (root/n).is_file()],\n                note=\'May contain partial runs or load failures. Check comparison_status.csv, n_ok/n_missing and model_errors.jsonl. Contains private transcripts; no original audio or weights exported.\')\n            archive.writestr(\'EXPORT_MANIFEST.json\',json.dumps(manifest,ensure_ascii=False,indent=2))\n        tmp.replace(target)\n    finally:\n        if tmp.exists(): tmp.unlink()\n    return str(target)\n'
exec(compile(BENCHMARK_SOURCE,"asr_benchmark.py","exec"))
BENCHMARK_SOURCE_SHA256 = '6a8b22f2efb62154beb2a82f03aa4272416e040c266a82bea878baa1e742adfb'
print("Pipeline:", CODE_VERSION)


## 4. Kiểm tra input và chọn audio
Ưu tiên cache follow-up đã sửa; không lấy metadata cũ từ audit. Nếu chỉ có ZIP, lấy đúng CSV/audio cần dùng, không giải nén environment hoặc toàn dataset.


In [ ]:
from IPython.display import display
CACHE_DIR, cache_provenance = resolve_followup_cache(FOLLOWUP_INPUT,WORK_DIR,CACHE_DIR)
references, joined, benchmark, cache_hashes = load_cache(CACHE_DIR)
rows = select_rows(joined,benchmark,STAGE)
print('Cache đang dùng:',CACHE_DIR)
print('Audit cũ:',AUDIT_INPUT,'— mount:',AUDIT_INPUT.is_dir(),'(không dùng thay follow-up)')
print('Tập đã chọn:',STAGE,len(rows),rows.kind.value_counts().to_dict())
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
if RESUME_DIR is not None:
    if not Path(RESUME_DIR).is_dir(): raise FileNotFoundError(RESUME_DIR)
    if any(OUTPUT_DIR.iterdir()): raise RuntimeError('Resume cần OUTPUT_DIR trống để tránh ghi đè.')
    shutil.copytree(RESUME_DIR,OUTPUT_DIR,dirs_exist_ok=True)
    RESUME_DIR=None
if RUN_INFERENCE:
    AUDIO_ROOT, audio_provenance = prepare_selected_audio(rows,DATASET_ROOT,WORK_DIR,DATASET_ZIP)
else:
    AUDIO_ROOT=DATASET_ROOT
    audio_provenance=dict(mode='not_checked_scoring_only')
write_json(OUTPUT_DIR/'input_provenance.json',dict(cache=cache_provenance,audio=audio_provenance,
    original_audit_mount=str(AUDIT_INPUT),followup_mount=str(FOLLOWUP_INPUT)))
(OUTPUT_DIR/'dependency_check.txt').write_text(globals().get('DEPENDENCY_CHECK','Không chạy cell pip check.'),encoding='utf-8')
rows[['ref_uid','sample_uid','kind','duration']].to_csv(OUTPUT_DIR/('selection_'+STAGE+'.csv'),index=False)
annotation_templates(rows,OUTPUT_DIR)
print('Audio root:',AUDIO_ROOT,'—',audio_provenance['mode'])
print('Input hợp lệ. Giữ nguyên split và ID cho mọi model.')


## 5. Kiểm tra GPU và thư viện
Dừng sớm nếu thiếu package/GPU; chưa tải checkpoint.


In [ ]:
if RUN_INFERENCE:
    import torch
    import transformers
    from transformers import WhisperProcessor, WhisperForConditionalGeneration, Wav2Vec2Processor, Wav2Vec2ForCTC, Qwen3ASRForConditionalGeneration, AutoProcessor
    import soundfile, psutil, huggingface_hub
    if DEVICE.startswith('cuda'):
        if not torch.cuda.is_available(): raise RuntimeError('Bật GPU trong Settings của Kaggle rồi chạy lại.')
        torch.cuda.get_device_properties(torch.device(DEVICE))
    elif DEVICE!='cpu': raise ValueError('DEVICE phải là cuda:0 hoặc cpu.')
    if DEVICE=='cpu' and PRECISION!='float32': raise ValueError('CPU phải dùng PRECISION=float32.')
    if PRECISION not in ['float16','float32']: raise ValueError('Precision chưa được hỗ trợ.')
    if transformers.__version__!='5.14.1': raise RuntimeError('Bản này cần Transformers 5.14.1 để chạy Qwen native. Chạy cell cài trong kernel mới.')
    env=runtime_info()
    write_json(OUTPUT_DIR/'runtime.json',env)
    print('GPU:',env['gpu'],'| torch:',env['versions']['torch'],'| transformers:',env['versions']['transformers'])
else:
    print('Chỉ chấm lại: không cần GPU hoặc tải checkpoint.')


## 6. Chuẩn bị lượt chạy
Lưu config, code và ID; mỗi model dùng cùng audio. Tự ghi lại bảng kết quả sau từng model để giữ tiến độ.


In [ ]:
cfg=dict(output_dir=str(OUTPUT_DIR),dataset_root=str(AUDIO_ROOT),stage=STAGE,
    device=DEVICE,precision=PRECISION,offline=OFFLINE,retry_errors=RETRY_ERRORS,
    max_new_tokens=MAX_NEW_TOKENS,source_sha256=BENCHMARK_SOURCE_SHA256)
selected=[c for c in CANDIDATES if c['name'] in SELECTED_MODELS]
if not selected or len(selected)!=len(SELECTED_MODELS): raise ValueError('SELECTED_MODELS có tên trùng/lạ hoặc rỗng.')
write_json(OUTPUT_DIR/'benchmark_request.json',dict(config=cfg,candidates=selected,
    selected_ref_uids=rows.ref_uid.tolist(),cache_hashes=cache_hashes,
    expected_audio_per_model=len(rows),normalization=NORMALIZATION,
    selection_constraints=dict(max_latency_p95_ms=MAX_LATENCY_P95_MS,max_vram_gb=MAX_VRAM_GB)))
(OUTPUT_DIR/'benchmark_source.py').write_text(BENCHMARK_SOURCE,encoding='utf-8')
run_dirs=[]
def refresh_results():
    global summary,scored,comparison_status
    summary,scored=evaluate(rows,OUTPUT_DIR,run_dirs=run_dirs)
    paired_bootstrap(scored,OUTPUT_DIR/'paired_bootstrap.csv')
    make_review(scored,OUTPUT_DIR)
    selection_table(summary,scored,OUTPUT_DIR/'model_selection.csv',MAX_LATENCY_P95_MS,MAX_VRAM_GB)
    status=[]
    for spec in selected:
        matches=[d for d in set(run_dirs) if json.loads((d/'config.json').read_text())['model_id']==spec['model_id']]
        if not matches:
            status.append(dict(model=spec['name'],model_id=spec['model_id'],run='',stage=STAGE,
                n_expected=len(rows),n_ok=0,n_error=0,n_missing=len(rows),state='not_run_or_checkpoint_error'))
        for d in matches:
            stats=aggregate(scored[d.name])
            state='ready' if stats['n_ok']==len(rows) else ('incomplete' if stats['n_missing'] else 'completed_with_errors')
            status.append(dict(model=spec['name'],model_id=spec['model_id'],run=d.name,stage=STAGE,
                n_expected=len(rows),n_ok=stats['n_ok'],n_error=stats['n_error'],n_missing=stats['n_missing'],state=state))
    comparison_status=pd.DataFrame(status)
    comparison_status.to_csv(OUTPUT_DIR/'comparison_status.csv',index=False)

def run_one(name):
    global run_dirs
    specs=[s for s in selected if s['name']==name]
    if not specs:
        print('Bỏ qua model không nằm trong SELECTED_MODELS:',name)
        return
    print('MODEL:',specs[0]['model_id'],'| stage:',STAGE,'| audio:',len(rows))
    if RUN_INFERENCE:
        current=run_benchmark(rows,specs,cfg,cache_hashes)
    else:
        current=[]
        for d in OUTPUT_DIR.iterdir():
            if not d.is_dir() or not (d/'config.json').is_file(): continue
            saved=json.loads((d/'config.json').read_text())
            if (saved.get('model_id')==specs[0]['model_id'] and saved.get('source_sha256')==cfg['source_sha256']
                and saved.get('cache_hashes')==cache_hashes): current.append(d)
    run_dirs=list(dict.fromkeys(run_dirs+current))
    refresh_results()
    display(comparison_status[comparison_status.model==name])
refresh_results()


## 7. PhoWhisper-medium — ứng viên chính
`vinai/PhoWhisper-medium`: chuyên biệt tiếng Việt, tăng năng lực so với small; greedy `vi/transcribe`.


In [ ]:
run_one('phowhisper_medium')


## 8. Whisper-large-v3-turbo — ứng viên chính
`openai/whisper-large-v3-turbo`: thử cân bằng chất lượng/tốc độ, không mặc định tốt nhất.


In [ ]:
run_one('whisper_turbo')


## 9. Qwen3-ASR-0.6B — ứng viên chính
`Qwen/Qwen3-ASR-0.6B-hf`: runtime native Transformers, tiếng Việt cố định, không context/hotword; tối đa 512 token. Dùng FP16 trên T4.


In [ ]:
run_one('qwen3_asr_06b')


## 10. Qwen3-ASR-1.7B — ứng viên chính
`Qwen/Qwen3-ASR-1.7B-hf`: so lợi ích chất lượng với chi phí của bản 0.6B trên cùng dữ liệu. Chưa đo VRAM/T4; OOM phải ghi lỗi.


In [ ]:
run_one('qwen3_asr_17b')


## 11. wav2vec2 tiếng Việt — ứng viên chính
`nguyenvulebinh/wav2vec2-base-vietnamese-250h`: so CTC cỡ nhỏ, greedy không LM ngoài. Giữ đủ audio >10 s và ghi cảnh báo; kiểm tra license CC-BY-NC-4.0 trước khi chốt triển khai.


In [ ]:
run_one('wav2vec2_vi')


## 12. Whisper-small — tùy chọn
Mặc định bỏ qua; thêm `whisper_small` vào SELECTED_MODELS nếu muốn đối chiếu kết quả cũ.


In [ ]:
run_one('whisper_small')


## 13. PhoWhisper-small — tùy chọn
Mặc định bỏ qua; thêm `phowhisper_small` nếu muốn so chi phí của small/medium.


In [ ]:
run_one('phowhisper_small')


## 14. PhoASR-whisper-small — tùy chọn
`Qualcomm-AI-Research/PhoASR-whisper-small`: model Việt mới, có hạn chế giọng miền Trung và license riêng. Thêm `phoasr_small` nếu quyền sử dụng phù hợp.


In [ ]:
run_one('phoasr_small')


## 15. So sánh để chọn model
`ready` = đủ prediction thành công, chưa phải kết luận model tốt. WER/CER càng thấp càng tốt; vẫn cần review tên/số/phủ định. Lỗi được chấm như câu rỗng; `n_ok=0` không có WER hợp lệ. CER bỏ khoảng trắng, giữ dấu; không đổi số chữ thành chữ số.
Bootstrap lấy mẫu theo hội thoại; CI chứa 0 chưa đủ kết luận thắng. Latency gồm đọc audio đến decode, tách load/warm-up; batch 1. RAM là RSS lấy mẫu, VRAM là PyTorch peak allocated/reserved, không phải toàn hệ thống.
`model_selection.csv`: loại run thiếu/lỗi hoặc vượt giới hạn bạn đặt, đánh dấu nhóm không bị model khác vượt cả WER lẫn latency. Chưa tự chọn người thắng: cần validation đầy đủ và review tên/số/phủ định. Model bị load lỗi không có dòng metric, xem comparison_status/model_errors.


In [ ]:
refresh_results()
display(comparison_status)
if not summary.empty:
    columns=['run','n','n_ok','n_error','n_missing','wer','cer_no_spaces','failure_rate',
             'latency_p50_ms','latency_p95_ms','rtf_weighted','rss_peak_bytes','vram_peak_allocated_bytes','vram_peak_reserved_bytes']
    display(summary.loc[summary.subset=='all',columns])
else:
    print('Chưa có run để chấm. Xem model_errors.jsonl hoặc chạy các cell model.')
selection=selection_table(summary,scored,OUTPUT_DIR/'model_selection.csv',MAX_LATENCY_P95_MS,MAX_VRAM_GB)
display(selection)
print('Smoke chỉ kiểm tra pipeline. Chạy benchmark 100, rồi validation 991 cho ứng viên cạnh tranh để chọn model.')
print('Chưa đủ model vẫn tải ZIP để phân tích lỗi; n_missing không phải kết quả thành công.')


## 16. Tạo ZIP và tải về
Chạy riêng cell này được nếu output còn trên đĩa. ZIP gồm prediction/reference, metrics, config/runtime, code, lỗi, annotation và checksum; loại audio, environment và weights. **Giữ private và gửi lại ZIP để phân tích.**
Annotation là biểu mẫu, chưa phải nhãn thật. Nghe trước khi xem prediction, dùng `uncertain` nếu chưa chắc; không suy vùng miền từ transcript. ASR không tự suy người nhận/thời gian/tool call.


In [ ]:
from pathlib import Path
from IPython.display import FileLink, display
import os
zip_source=Path(globals().get('OUTPUT_DIR','/kaggle/working/lifemate_asr_benchmark'))
def package_outputs(out_dir,zip_base):
    """Export comparison artifacts only; standalone and usable after interrupted runs."""
    from pathlib import Path
    import hashlib, json, zipfile
    root=Path(out_dir).resolve()
    if not root.is_dir(): raise FileNotFoundError(root)
    target=Path(str(zip_base)+'.zip').resolve()
    target.parent.mkdir(parents=True,exist_ok=True)
    shared={'runtime.json','selection.json','benchmark_request.json','benchmark_source.py','input_provenance.json',
            'scoring_patch.json','scoring_source.py',
            'metrics.csv','paired_bootstrap.csv','comparison_status.csv','model_selection.csv','dependency_check.txt',
            'dialect_annotations.csv','critical_gold_annotation.csv',
            'critical_model_review_template.csv','model_errors.jsonl'}
    run_files={'config.json','model_config.json','generation_config.json',
               'predictions.jsonl','per_audio.csv','recovery.jsonl'}
    selected=[]
    for p in sorted(root.iterdir()):
        if p.is_symlink(): continue
        if p.is_file() and (p.name in shared or
                (p.name.startswith('selection_') and p.suffix=='.csv') or
                p.name.endswith('_checkpoint.json')):
            selected.append(p)
        elif p.is_dir() and not p.name.endswith('_environment') and (p/'config.json').is_file():
            for f in sorted(p.iterdir()):
                if f.is_symlink() or not f.is_file(): continue
                if f.name in run_files or (f.name.startswith(('session_','warmup_')) and f.suffix=='.json') or f.name.startswith('predictions.jsonl.torn-'):
                    selected.append(f)
    entries=[]
    tmp=target.with_suffix('.zip.tmp')
    try:
        with zipfile.ZipFile(tmp,'w',compression=zipfile.ZIP_DEFLATED) as archive:
            for f in selected:
                relative=f.relative_to(root).as_posix()
                data=f.read_bytes()
                entries.append(dict(path=relative,bytes=len(data),sha256=hashlib.sha256(data).hexdigest()))
                archive.writestr(relative,data)
            manifest=dict(schema_version=1,private=True,files=entries,
                missing_summary_files=[n for n in ['benchmark_request.json','metrics.csv','comparison_status.csv'] if not (root/n).is_file()],
                note='May contain partial runs or load failures. Check comparison_status.csv, n_ok/n_missing and model_errors.jsonl. Contains private transcripts; no original audio or weights exported.')
            archive.writestr('EXPORT_MANIFEST.json',json.dumps(manifest,ensure_ascii=False,indent=2))
        tmp.replace(target)
    finally:
        if tmp.exists(): tmp.unlink()
    return str(target)

zip_path=package_outputs(zip_source,zip_source.parent/'lifemate_asr_benchmark_outputs')
print('Tải ZIP này, giữ private:',zip_path)
display(FileLink(os.path.relpath(zip_path,Path.cwd())))
